# Prática 01/12 · Tribunal do Pipeline: auditar e consertar um experimento
**Aprendizagem Computacional · 4º TIA · Aula 06 · versão INICIAL**

Hoje é o ensaio da banca. Um grupo fictício, o **Grupo Réu**, usou a mesma base do PPI-molde e chegou a um resultado espetacular: R² de 1, o modelo "acerta tudo". Vocês são o tribunal:

1. rodam o experimento do réu e veem o resultado "excelente";
2. apontam os **crimes** de método (do catálogo da parte 3);
3. consertam um por um;
4. refazem o experimento honesto e comparam;
5. ensaiam as 7 perguntas da banca.

Os mesmos crimes aparecem, com frequência, em projetos reais. Auditem o notebook do grupo de vocês com a mesma lista antes de 01/12.

| TODO | Tarefa | Apostila |
|---|---|---|
| 1 | Marcar os crimes do réu no catálogo | cap. 2, 5, 9 |
| 2 | Tirar de X o que causa vazamento | cap. 5 |
| 3 | Split reprodutível, antes de tudo | cap. 9 |
| 4 | Comparar candidatos e réguas com validação cruzada, escala dentro do `Pipeline` | cap. 9, 12, 13 |
| 5 | Escolher pela CV e abrir o teste uma vez | cap. 9, 24 |

## 1 · Carregar a base (pronto)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
SEMENTE = 42
pd.set_option('display.precision', 2)

ARQUIVO = 'base_analitica.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-12-01_Aula06_Tribunal_do_Pipeline/dados/' + ARQUIVO
candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
base = pd.read_csv(local if local is not None else URL, dtype={'CD_MUNICIPIO': str})
print('Base lida de:', local.resolve() if local is not None else URL, '|', base.shape)

## 2 · O experimento do Grupo Réu (pronto, não altere)
Rode a célula **duas vezes** e compare os números.

In [ ]:
# ===== Notebook do Grupo Réu =====
base['COMPARECIMENTO_PCT'] = 100 * (base['QT_APTOS'] - base['QT_ABSTENCOES']) / base['QT_APTOS']   # "do mesmo arquivo do TSE"
COLS_REU = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO', 'ABST_2022_PCT',
            'CRESC_ELEITORADO_PCT', 'DIST_CUIABA_KM', 'QT_APTOS', 'QT_ABSTENCOES', 'COMPARECIMENTO_PCT']
escala = StandardScaler().fit(base[COLS_REU])                  # padroniza tudo de uma vez
Xs = escala.transform(base[COLS_REU])
X_tr, X_te, y_tr, y_te = train_test_split(Xs, base['ABSTENCAO_PCT'], test_size=0.2)

modelos_reu = {'Linear': LinearRegression(), 'Árvore': DecisionTreeRegressor(),
               'Random Forest': RandomForestRegressor(n_estimators=300)}
r2_reu = {nome: r2_score(y_te, m.fit(X_tr, y_tr).predict(X_te)) for nome, m in modelos_reu.items()}
escolhido_reu = max(r2_reu, key=r2_reu.get)
print({n: round(v, 3) for n, v in r2_reu.items()})
print(f'Modelo escolhido: {escolhido_reu}, com R² = {r2_reu[escolhido_reu]:.3f} no teste. Excelente!')

## 3 · TODO 1: o veredito
Quais destes crimes o Grupo Réu cometeu? Cuidado: o catálogo tem acusações falsas.

| Código | Acusação |
|---|---|
| A | Há em X variáveis que só existem **depois** da eleição e com as quais se calcula y. |
| B | A escala foi ajustada na base **toda**, antes de separar o teste. |
| C | O split não fixa `random_state`: o resultado muda a cada execução. |
| D | O modelo foi **escolhido** pelo resultado no teste. |
| E | Não há baseline: o R² aparece sem nenhuma régua para comparar. |
| F | Usaram regressão, mas o problema é de classificação. |
| G | A base tem municípios repetidos. |
| H | A Random Forest foi usada sem escala, e isso é obrigatório. |

Escreva os códigos dos crimes **verdadeiros** no conjunto abaixo, por exemplo `{'A', 'B'}`.

In [ ]:
crimes = set()   # TODO 1: códigos dos crimes verdadeiros, por exemplo {'A', 'B'}

**Por que as falsas são falsas?** F: a abstenção é um número contínuo, então regressão é a tarefa certa. G: a base tem uma linha por município (o `assert` do ETL garante). H: árvores e florestas comparam valores de **uma** variável por vez, então não precisam de escala; k-NN e regressão com regularização é que precisam.

## 4 · TODO 2: tirar o vazamento de X (cap. 5)
`QT_ABSTENCOES` e `QT_APTOS` são contagens **de 2026**: y é literalmente 100 × uma ÷ a outra. E `COMPARECIMENTO_PCT` é 100 − y. Com elas em X, o modelo "prevê" a abstenção fazendo a conta. É por isso que o R² do réu dá 1. E o modelo seria inútil no dia de usar, porque essas colunas só existem depois da eleição.

Monte `X_COLS` só com as 7 variáveis conhecidas antes da eleição: a lista do réu **sem** as três colunas de 2026.

In [ ]:
X_COLS = None   # TODO 2: a lista COLS_REU sem 'QT_APTOS', 'QT_ABSTENCOES' e 'COMPARECIMENTO_PCT'
assert not {'QT_APTOS', 'QT_ABSTENCOES', 'COMPARECIMENTO_PCT'} & set(X_COLS)
X, y = base[X_COLS], base['ABSTENCAO_PCT']

## 5 · TODO 3: o split honesto (cap. 9)
Separe 20% para teste **antes de qualquer ajuste**, com `random_state=SEMENTE`, sobre os dados **originais** (sem escala). A escala vai para dentro do `Pipeline` no passo seguinte.

In [ ]:
X_treino, X_teste, y_treino, y_teste = None, None, None, None   # TODO 3: train_test_split com random_state
print('Treino:', len(X_treino), '| Teste (guardado):', len(X_teste))

## 6 · TODO 4: comparar com réguas, por validação cruzada (cap. 9, 12 e 13)
Os candidatos são os mesmos do réu, mais as duas réguas do molde. A escala fica **dentro** do `Pipeline` da linear, e todos passam pelas mesmas 5 dobras **do treino**.
Dentro do `for`, chame `cross_validate(modelo, X_treino, y_treino, cv=cv, scoring='neg_mean_absolute_error')`.

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.dummy import DummyRegressor

class Persistencia(BaseEstimator, RegressorMixin):
    """Baseline "a abstenção se repete": abstenção de 2022 + a variação média vista no treino."""
    def __init__(self, coluna='ABST_2022_PCT'):
        self.coluna = coluna
    def fit(self, X, y):
        self.desloc_ = float(np.mean(np.asarray(y) - X[self.coluna].to_numpy()))
        return self
    def predict(self, X):
        return X[self.coluna].to_numpy() + self.desloc_

cv = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)
candidatos = {
    'Baseline: média': DummyRegressor(strategy='mean'),
    'Baseline: persistência': Persistencia(),
    'Linear': Pipeline([('escala', StandardScaler()), ('modelo', LinearRegression())]),
    'Árvore (prof. 4)': DecisionTreeRegressor(max_depth=4, random_state=SEMENTE),
    'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=SEMENTE),
}
linhas = []
for nome, modelo in candidatos.items():
    r = None   # TODO 4: cross_validate(...) no treino, com o cv e scoring='neg_mean_absolute_error'
    linhas.append({'modelo': nome, 'MAE_cv': -r['test_score'].mean(), 'desvio': r['test_score'].std()})
tabela_cv = pd.DataFrame(linhas).set_index('modelo').sort_values('MAE_cv')
tabela_cv

## 7 · TODO 5: escolher pela CV e abrir o teste uma vez (cap. 9 e 24)
Escolha o candidato de **menor MAE na validação cruzada** (`tabela_cv.index[0]`), ajuste-o em todo o treino e meça o teste. As réguas também são medidas, para o veredito final.

In [ ]:
escolhido = None   # TODO 5: o nome do candidato de menor MAE na CV (primeira linha de tabela_cv)
final = candidatos[escolhido].fit(X_treino, y_treino)
veredito = pd.DataFrame({n: {'MAE_teste': mean_absolute_error(y_teste, m.fit(X_treino, y_treino).predict(X_teste)),
                             'R2_teste': r2_score(y_teste, m.predict(X_teste))}
                         for n, m in {escolhido: final, 'Baseline: persistência': candidatos['Baseline: persistência'],
                                      'Baseline: média': candidatos['Baseline: média']}.items()}).T
print('Escolhido pela CV:', escolhido)
veredito

**Pergunta para discussão:** o réu tinha R² de 1; o experimento honesto tem 0,77. O resultado "piorou"?

**Resposta esperada:** não. O R² do réu media a capacidade de refazer uma conta com dados que só existem depois da eleição; ele seria inútil no dia de usar. O 0,77 mede o que o modelo faz com o que se sabe **antes**, e ainda vence com folga as duas réguas. Um número menor e honesto vale mais que um número alto que não se sustenta na banca.

## 8 · Ensaio da banca (em dupla, 10 min)
Um faz a pergunta, o outro responde **sem ler**, em até 30 segundos. Depois troquem. Façam o mesmo com o projeto do grupo de vocês.

| Pergunta | Resposta esperada no PPI-molde |
|---|---|
| 1. Qual é o y? | A abstenção de 2026 (%) de cada município, no 1º turno para presidente. |
| 2. Qual é o baseline? | Dois: prever a média e repetir 2022 corrigido (persistência). |
| 3. Por que essa métrica? | MAE em p.p.: o gestor entende "erra 1,2 ponto", e poucos municípios extremos não dominam. |
| 4. Onde foi feito o split? | Antes de tudo, 80/20 com `random_state=42`; escolha por validação cruzada no treino; teste aberto uma vez. |
| 5. Há vazamento? | Não: só entram em X variáveis conhecidas antes da eleição, e a escala fica dentro do `Pipeline`. |
| 6. A conclusão tem evidência? | MAE de 1,23 no teste, contra 1,96 da persistência e 2,13 da média; R² de 0,77. |
| 7. Roda do zero? | Sim: o ETL refaz a base a partir do TSE, e o notebook roda de cima para baixo no Colab. |

## Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None ou ...) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('veredito: os 5 crimes certos, sem acusação falsa', lambda: crimes == {'A', 'B', 'C', 'D', 'E'}),
    conferir('X sem as contagens de 2026 (7 variáveis)', lambda: len(X_COLS) == 7 and not {'QT_APTOS', 'QT_ABSTENCOES', 'COMPARECIMENTO_PCT'} & set(X_COLS)),
    conferir('split reprodutível: 112 no treino e 29 no teste', lambda: len(X_treino) == 112 and len(X_teste) == 29),
    conferir('CV: linear com MAE 1,33', lambda: round(tabela_cv.loc['Linear', 'MAE_cv'], 2) == 1.33),
    conferir('CV: persistência com MAE 1,78', lambda: round(tabela_cv.loc['Baseline: persistência', 'MAE_cv'], 2) == 1.78),
    conferir('escolha pela CV: Linear', lambda: escolhido == 'Linear'),
    conferir('teste: MAE 1,23 do escolhido', lambda: round(veredito.loc[escolhido, "MAE_teste"], 2) == 1.23),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist final do grupo (antes de 01/12)
Passem o notebook do grupo por esta lista, como fizeram com o réu:
- [ ] Nenhuma coluna de X é calculada a partir de y ou só existe depois do fato que se quer prever.
- [ ] Toda transformação que aprende com os dados (escala, imputação, PCA) está dentro de um `Pipeline`.
- [ ] O split tem `random_state` e vem antes de qualquer ajuste.
- [ ] Há pelo menos um baseline na mesma tabela do modelo.
- [ ] O modelo foi escolhido pela validação cruzada; o teste foi aberto uma vez.
- [ ] Outra pessoa rodou o notebook do zero, seguindo só o README.